# Diviser pour régner — Exercice 1 : les mots mystérieux (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** lecture de fichier, recherche **séquentielle** vs recherche **dichotomique**, coût d'un algorithme.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

> **Principe (cours)** — La méthode « diviser pour régner » (*divide and conquer*) consiste à « diviser » un problème en sous-problèmes indépendants dont les solutions sont recombinées :
> - on décompose le problème en sous-problèmes ;
> - on résout chaque sous-problème ;
> - on fusionne les solutions des sous-problèmes en une solution globale.

Les fichiers `dico.txt` et `mots.txt` sont ceux distribués avec le cours. Ils sont placés à côté de ce notebook.

In [1]:
import time

---
# QUESTION 1
## 1 (a)

#### 📝 ÉNONCÉ

> Le fichier `dico.txt` contient une liste de mots de la langue française, triés dans l'ordre lexicographique.
> Télécharger ce fichier et exécuter le code suivant pour créer une liste contenant les mots qu'il contient.
> ```python
> f = open("dico.txt", "r", encoding="utf-8")
>
> dico = []
> for ligne in f:
>     dico.append(ligne[:-1])
> ```

#### ✅ RÉSOLUTION

Le code de l'énoncé (on ajoute `f.close()` pour fermer le fichier) :

In [2]:
f = open("dico.txt", "r", encoding="utf-8")

dico = []
for ligne in f:
    dico.append(ligne[:-1])
f.close()

#### 💡 EXPLICATION PÉDAGOGIQUE

- `open(..., "r", encoding="utf-8")` ouvre le fichier en **lecture**. L'encodage UTF-8 est indispensable pour lire correctement les accents (é, è, ç...).
- `for ligne in f` parcourt le fichier **ligne par ligne**. Chaque ligne se termine par le caractère de retour à la ligne `"\n"`.
- `ligne[:-1]` enlève le **dernier caractère**, c'est-à-dire ce `"\n"`.

⚠️ **Piège, qui se produit vraiment avec ce fichier :** la dernière ligne de `dico.txt` ne se termine pas par `"\n"`. Pour elle, `ligne[:-1]` supprime la dernière **lettre** du mot : le dernier mot `"ôtés"` devient `"ôté"` (la cellule suivante le montre). La version plus sûre est `ligne.rstrip("\n")`, ou `ligne.strip()`.

## 1 (b)

#### 📝 ÉNONCÉ

> Combien de mots ce fichier contient-il ?

#### ✅ RÉSOLUTION

C'est la longueur de la liste : `len(dico)`, soit **142 694 mots**.

In [3]:
print(len(dico), "mots dans le dictionnaire")
print("Premiers mots :", dico[:6])
print("Derniers mots :", dico[-3:])

142694 mots dans le dictionnaire
Premiers mots : ['a', 'a', 'a', 'a capella', 'a cappella', 'a contrario']
Derniers mots : ['ôtées', 'ôtés', 'ôté']


#### 💡 EXPLICATION PÉDAGOGIQUE

- `len(dico)` compte les **lignes** du fichier : 142 694.
- Certains mots apparaissent **plusieurs fois** (par exemple `"a"` trois fois) et certaines lignes contiennent plusieurs mots (`"a capella"`). Si on veut les mots **différents**, `len(set(dico))` donne 125 653. La réponse attendue est `len(dico)`.
- Le dernier mot affiché est `"ôté"` au lieu de `"ôtés"` : c'est le piège de `ligne[:-1]` expliqué juste au-dessus.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> On fournit également un fichier `mots.txt` qui contient 20000 mots. À l'exception de quelques uns, tous ces mots sont contenus dans le fichier `dico.txt`.
> Écrire un script python qui permet d'afficher ces mystérieux mots en moins de deux secondes.

## Première idée (trop lente) : la recherche séquentielle

In [4]:
with open("mots.txt", encoding="utf-8") as f:
    mots = [ligne.rstrip("\n") for ligne in f]

debut = time.time()
for mot in mots[:200]:
    if mot not in dico:
        pass
duree = time.time() - debut
print(f"200 mots : {duree:.2f} s  ->  estimation pour {len(mots)} mots : {duree * len(mots) / 200:.0f} s")

200 mots : 0.12 s  ->  estimation pour 19999 mots : 12 s


#### 💡 EXPLICATION PÉDAGOGIQUE

`mot in dico` parcourt la liste **du début jusqu'à trouver le mot**. Pour un dictionnaire de $N$ mots, cela fait en moyenne $N/2$ comparaisons, et $N$ pour un mot absent.

Pour 20 000 mots et $N \approx 140\,000$ : environ $20\,000 \times 70\,000 = 1{,}4 \times 10^9$ comparaisons, **beaucoup trop**. On le vérifie en chronométrant seulement les **200 premiers mots** : l'estimation dépasse largement 2 secondes.

(Le fichier `mots.txt` contient en fait 19 999 lignes, et non 20 000 : cela ne change rien à la méthode.)

## La bonne idée : la recherche dichotomique

#### ✅ RÉSOLUTION

**Étape 1 :** la fonction de recherche dichotomique.

In [5]:
def recherche_dichotomique(t, x):
    # renvoie True si x est dans le tableau TRIÉ t
    gauche, droite = 0, len(t) - 1
    while gauche <= droite:
        milieu = (gauche + droite) // 2
        if t[milieu] == x:
            return True
        elif t[milieu] < x:
            gauche = milieu + 1      # x est forcément à droite du milieu
        else:
            droite = milieu - 1      # x est forcément à gauche du milieu
    return False


assert recherche_dichotomique([2, 5, 7, 9, 12], 9)
assert not recherche_dichotomique([2, 5, 7, 9, 12], 8)
assert not recherche_dichotomique([], 8)

#### ✅ RÉSOLUTION

**Étape 2 :** le script qui affiche les mots mystérieux.

In [6]:
def est_trie(t):
    for i in range(len(t) - 1):
        if t[i] > t[i + 1]:
            return False
    return True


debut = time.time()

if not est_trie(dico):
    print("Le dictionnaire n'est pas trié dans l'ordre de Python : on le trie.")
    dico = sorted(dico)

mysteres = [mot for mot in mots if not recherche_dichotomique(dico, mot)]

print("Mots mystérieux :", mysteres)
print(f"Temps total : {time.time() - debut:.3f} s")

Le dictionnaire n'est pas trié dans l'ordre de Python : on le trie.
Mots mystérieux : ['carabistouille', "c'est", 'tro', 'leent', 'vraimeeent', 'Troooop', 'leeeeent', '¯\\_(ツ)_/¯']
Temps total : 0.046 s


#### 💡 EXPLICATION PÉDAGOGIQUE

**Principe de la dichotomie.** Le dictionnaire est **trié**. Pour chercher un mot :
1. on regarde le mot **au milieu** de la zone de recherche ;
2. s'il est égal au mot cherché, c'est gagné ;
3. s'il est **plus grand**, le mot cherché ne peut être que dans la **moitié gauche**, sinon dans la **moitié droite** ;
4. on recommence sur la moitié restante, jusqu'à ce que la zone soit vide.

À chaque étape, la zone de recherche est **divisée par 2**. Pour $N \approx 140\,000$, il faut au plus environ $\log_2(140\,000) \approx 18$ comparaisons, au lieu de 140 000.

⚠️ **Point essentiel : la dichotomie n'est correcte que si le tableau est trié dans le même ordre que celui de Python.** Le fichier est trié « dans l'ordre lexicographique » de la langue française. Python, lui, compare les caractères selon leur code Unicode, et place par exemple `"é"` **après** `"z"`. Si les deux ordres diffèrent, la dichotomie peut « rater » des mots présents. On vérifie donc d'abord avec `est_trie` (c'est aussi la question 1 de l'exercice 2) et, si besoin, on **trie une seule fois** le dictionnaire avec `sorted`.

Avec ce fichier, il est bien rangé dans l'ordre de Python, **sauf à la toute fin** : à cause du piège de `ligne[:-1]`, le dernier mot `"ôtés"` est devenu `"ôté"`, qui devrait être placé avant `"ôtés"`. C'est pour cela que `est_trie` répond `False` et que le script trie. En lisant le fichier avec `ligne.rstrip("\n")`, la liste serait triée et le tri inutile.

**Les mots mystérieux** forment une phrase : *« carabistouille, c'est tro leent vraimeeent Troooop leeeeent ¯\\_(ツ)_/¯ »*. C'est un clin d'œil à la recherche séquentielle, beaucoup trop lente ! (« carabistouille » est un vrai mot, mais il n'est pas dans `dico.txt`.)

**Comparaison des coûts :**

| Méthode | Comparaisons par mot | Total pour 20 000 mots |
|:---|:---|:---|
| Recherche séquentielle | $\approx N$ ≈ 140 000 | ≈ 3 milliards |
| Recherche dichotomique | $\approx \log_2(N)$ ≈ 18 | ≈ 360 000 |

**Pour aller plus loin :** le module `bisect` fournit la dichotomie toute faite, et un `set` permet de tester l'appartenance en temps quasi constant. Ce n'est pas l'objet de l'exercice, qui demande d'utiliser l'idée « diviser pour régner ».